``` text
输入 Token IDs: x [B, T]
        │
  ┌─────┴──────────────────┐
  │ Token & Pos Embeddings │ ──> wte(x) + wpe(pos)
  └─────┬──────────────────┘
        │  x: [B, T, C]
        ▼
  ┌────────────────────────┐ <───┐
  │  LayerNorm (Pre-LN)    │     │
  ├────────────────────────┤     │
  │ Causal Self-Attention  │     │ 循环 N 层
  ├────────────────────────┤     │ (Transformer Block)
  │   Residual Add (+)     │ ────┤
  ├────────────────────────┤     │
  │  LayerNorm (Pre-LN)    │     │
  ├────────────────────────┤     │
  │     MLP / FFN          │     │
  ├────────────────────────┤     │
  │   Residual Add (+)     │ ────┘
  └─────┬──────────────────┘
        │  x: [B, T, C]
        ▼
  ┌────────────────────────┐
  │  LayerNorm (Final LN)  │
  └─────┬──────────────────┘
        │
  ┌─────┴──────────────────┐
  │  LM Head (Linear)      │ ──> [B, T, Vocab_Size]
  └────────────────────────┘
```

符号：
- B：batch size
- T：序列长度
- C：隐藏维度 n_embd
- Vocab_Size：词表大小
- N：Transformer Block 数量

每个 GPT Block：
$$
x'=x+\operatorname{Attention}(\operatorname{LN}(x))
$$
$$
x''=x'+\operatorname{MLP}(\operatorname{LN}(x'))
$$

这是 Pre-LN 结构。


In [7]:
import math
from dataclasses import dataclass

import torch
import torch.nn as nn
import torch.nn.functional as F


@dataclass
class GPTConfig:
    vocab_size: int
    block_size: int = 256
    n_layer: int = 4
    n_head: int = 4
    n_embd: int = 256
    dropout: float = 0.1

    def __post_init__(self):
        if self.n_embd % self.n_head != 0:
            raise ValueError("n_embd 必须能够被 n_head 整除")


class CausalSelfAttention(nn.Module):
    def __init__(self, config: GPTConfig):
        super().__init__()

        self.n_head = config.n_head
        self.n_embd = config.n_embd
        self.head_dim = config.n_embd // config.n_head

        # 一次性生成 Q、K、V
        self.qkv_proj = nn.Linear(
            config.n_embd,
            3 * config.n_embd,
            bias=False,
        )

        self.out_proj = nn.Linear(
            config.n_embd,
            config.n_embd,
            bias=False,
        )

        self.attn_dropout = nn.Dropout(config.dropout)
        self.resid_dropout = nn.Dropout(config.dropout)

        # [1, 1, block_size, block_size]
        causal_mask = torch.tril(
            torch.ones(config.block_size, config.block_size)
        )

        self.register_buffer(
            "causal_mask",
            causal_mask.view(
                1,
                1,
                config.block_size,
                config.block_size,
            ),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        B, T, C = x.shape

        # [B, T, C] -> [B, T, 3C]
        qkv = self.qkv_proj(x)
        q, k, v = qkv.chunk(3, dim=-1)

        # [B, T, C] -> [B, H, T, head_dim]
        q = q.view(
            B, T, self.n_head, self.head_dim
        ).transpose(1, 2)

        k = k.view(
            B, T, self.n_head, self.head_dim
        ).transpose(1, 2)

        v = v.view(
            B, T, self.n_head, self.head_dim
        ).transpose(1, 2)

        # [B, H, T, T]
        scores = q @ k.transpose(-2, -1)
        scores = scores / math.sqrt(self.head_dim)

        mask = self.causal_mask[:, :, :T, :T]

        scores = scores.masked_fill(
            mask == 0,
            torch.finfo(scores.dtype).min,
        )

        attention = F.softmax(scores, dim=-1)
        attention = self.attn_dropout(attention)

        # [B, H, T, head_dim]
        y = attention @ v

        # [B, H, T, head_dim] -> [B, T, C]
        y = y.transpose(1, 2).contiguous()
        y = y.view(B, T, C)

        y = self.out_proj(y)
        y = self.resid_dropout(y)

        return y


class MLP(nn.Module):
    def __init__(self, config: GPTConfig):
        super().__init__()

        self.net = nn.Sequential(
            nn.Linear(config.n_embd, 4 * config.n_embd),
            nn.GELU(),
            nn.Linear(4 * config.n_embd, config.n_embd),
            nn.Dropout(config.dropout),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)


class GPTBlock(nn.Module):
    def __init__(self, config: GPTConfig):
        super().__init__()

        self.ln1 = nn.LayerNorm(config.n_embd)
        self.attention = CausalSelfAttention(config)

        self.ln2 = nn.LayerNorm(config.n_embd)
        self.mlp = MLP(config)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # Pre-LN + 残差连接
        x = x + self.attention(self.ln1(x))
        x = x + self.mlp(self.ln2(x))

        return x


class MiniGPT(nn.Module):
    def __init__(self, config: GPTConfig):
        super().__init__()

        self.config = config

        self.token_embedding = nn.Embedding(
            config.vocab_size,
            config.n_embd,
        )

        self.position_embedding = nn.Embedding(
            config.block_size,
            config.n_embd,
        )

        self.dropout = nn.Dropout(config.dropout)

        self.blocks = nn.ModuleList([
            GPTBlock(config)
            for _ in range(config.n_layer)
        ])

        self.final_norm = nn.LayerNorm(config.n_embd)

        self.lm_head = nn.Linear(
            config.n_embd,
            config.vocab_size,
            bias=False,
        )

        # GPT 常用的 Embedding 与输出层权重共享
        self.lm_head.weight = self.token_embedding.weight

        self.apply(self._init_weights)

    @staticmethod
    def _init_weights(module: nn.Module):
        if isinstance(module, nn.Linear):
            nn.init.normal_(
                module.weight,
                mean=0.0,
                std=0.02,
            )

            if module.bias is not None:
                nn.init.zeros_(module.bias)

        elif isinstance(module, nn.Embedding):
            nn.init.normal_(
                module.weight,
                mean=0.0,
                std=0.02,
            )

    def forward(
        self,
        input_ids: torch.Tensor,
        labels: torch.Tensor | None = None,
    ):
        B, T = input_ids.shape

        if T > self.config.block_size:
            raise ValueError(
                f"序列长度 {T} 超过最大长度 "
                f"{self.config.block_size}"
            )

        positions = torch.arange(
            T,
            device=input_ids.device,
        )

        token_emb = self.token_embedding(input_ids)
        position_emb = self.position_embedding(positions)

        x = self.dropout(token_emb + position_emb)

        for block in self.blocks:
            x = block(x)

        x = self.final_norm(x)

        # [B, T, C] -> [B, T, V]
        logits = self.lm_head(x)

        loss = None

        if labels is not None:
            # 位置 t 的输出预测位置 t+1 的 token
            shift_logits = logits[:, :-1, :].contiguous()
            shift_labels = labels[:, 1:].contiguous()

            loss = F.cross_entropy(
                shift_logits.view(
                    -1,
                    self.config.vocab_size,
                ),
                shift_labels.view(-1),
                ignore_index=-100,
            )

        return logits, loss

In [8]:
config = GPTConfig(
    vocab_size=5000,
    block_size=128,
    n_layer=4,
    n_head=4,
    n_embd=256,
)

model = MiniGPT(config)

input_ids = torch.randint(
    low=0,
    high=config.vocab_size,
    size=(8, 64),
)

logits, loss = model(
    input_ids=input_ids,
    labels=input_ids,
)

print(logits.shape)
print(loss)

torch.Size([8, 64, 5000])
tensor(8.5879, grad_fn=<NllLossBackward0>)
